# Runtime governance

Generated from the book sources. Do not edit by hand: changes belong in the `.qmd` chapter.

> Setup the book does not print. Later cells depend on the state it creates, so run it.

In [ ]:
import os, json
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI()
CHAT_MODEL = os.environ["CHAT_MODEL"]

ORDERS = {
    "A-1001": {"refund_issued": True, "refund_date": "2026-06-24",
               "amount_eur": 79.90},
    "A-1002": {"refund_issued": False,
               "reason": "returned item not yet received",
               "amount_eur": 79.90},
}

def get_refund_status(order_id: str):
    return ORDERS.get(order_id, {"error": f"unknown order id '{order_id}'"})

def issue_refund(order_id: str, amount_eur: float):
    return {"status": "refund issued", "order_id": order_id,
            "amount_eur": amount_eur}

TOOL_IMPLEMENTATIONS = {
    "get_refund_status": get_refund_status,
    "issue_refund": issue_refund,
}

status_tool = {
  "type": "function",
  "function": {
    "name": "get_refund_status",
    "description": "Look up the refund status of an order.",
    "parameters": {
      "type": "object",
      "properties": {"order_id": {"type": "string"}},
      "required": ["order_id"],
      "additionalProperties": False}}}

refund_tool = {
  "type": "function",
  "function": {
    "name": "issue_refund",
    "description": "Issue a refund for an order.",
    "parameters": {
      "type": "object",
      "properties": {"order_id": {"type": "string"},
                     "amount_eur": {"type": "number"}},
      "required": ["order_id", "amount_eur"],
      "additionalProperties": False}}}

### A deterministic policy gate: allowlist, argument bounds, approval threshold, and step budget

`lst-policy-gate`

In [ ]:
POLICY = {
    "allowed_tools": {"get_refund_status", "issue_refund"},
    "refund_auto_limit_eur": 50.00,
    "max_steps": 6,
}

def gate(tool_name: str, args: dict):
    if tool_name not in POLICY["allowed_tools"]:
        return "deny", f"tool '{tool_name}' is not allowlisted"
    if tool_name == "issue_refund":
        amount = args.get("amount_eur", 0)
        if not 0 < amount <= 10_000:
            return "deny", f"amount {amount} outside valid bounds"
        if amount > POLICY["refund_auto_limit_eur"]:
            return ("escalate",
                    f"refund of {amount:.2f} EUR exceeds the "
                    f"{POLICY['refund_auto_limit_eur']:.2f} EUR "
                    "auto-approval limit; human approval required")
    return "allow", "within policy"

### The agent loop with the gate between the model's decision and the tool's execution

`lst-governed-loop`

In [ ]:
AUDIT_LOG = []

def run_governed_agent(user_input: str):
    messages = [
        {"role": "system", "content": (
            "You are a customer support agent. Use tools to check and "
            "issue refunds. If a tool reports that approval is "
            "required, tell the customer their request was escalated "
            "to a human reviewer.")},
        {"role": "user", "content": user_input},
    ]
    for step in range(POLICY["max_steps"]):
        resp = client.chat.completions.create(
            model=CHAT_MODEL, messages=messages,
            tools=[status_tool, refund_tool], tool_choice="auto",
            temperature=0.2, seed=42)
        msg = resp.choices[0].message
        if not msg.tool_calls:
            return msg.content
        messages.append(
            {"role": "assistant", "tool_calls": msg.tool_calls})
        for tc in msg.tool_calls:
            args = json.loads(tc.function.arguments)
            decision, reason = gate(tc.function.name, args)
            AUDIT_LOG.append({"step": step + 1,
                              "tool": tc.function.name, "args": args,
                              "decision": decision, "reason": reason})
            if decision == "allow":
                result = TOOL_IMPLEMENTATIONS[tc.function.name](**args)
            else:
                result = {"status": f"blocked ({decision})",
                          "reason": reason}
            print(f"Step {step + 1}: {tc.function.name}({args})"
                  f" -> {decision}")
            messages.append({"role": "tool", "tool_call_id": tc.id,
                             "content": json.dumps(result)})
    return "Step budget exhausted; escalating to a human agent."

### A run in which the gate escalates an over-threshold refund

`lst-gate-run`

In [ ]:
answer = run_governed_agent(
    "The item from order A-1002 arrived broken. "
    "Please refund the 79.90 EUR now.")
print("\nFinal answer:", answer)
print("\nAudit log:")
for entry in AUDIT_LOG:
    print(f"  {entry['decision']:9s} {entry['tool']}"
          f"({entry['args']}) - {entry['reason']}")

### A minimal span tracer: a run becomes a tree of named, timed, attributed operations

`lst-minimal-tracer`

In [ ]:
import time
from contextlib import contextmanager

class Tracer:
    def __init__(self):
        self.spans, self._stack, self._n = [], [], 0

    @contextmanager
    def span(self, name, **attributes):
        self._n += 1
        s = {"id": self._n,
             "parent": self._stack[-1]["id"] if self._stack else None,
             "name": name, "attrs": attributes,
             "t0": time.perf_counter()}
        self._stack.append(s)
        self.spans.append(s)
        try:
            yield s
        finally:
            s["ms"] = round((time.perf_counter() - s.pop("t0")) * 1000)
            self._stack.pop()

    def tree(self, parent=None, depth=0):
        for s in [x for x in self.spans if x["parent"] == parent]:
            attrs = " ".join(f"{k}={v}" for k, v in s["attrs"].items())
            print(f"{'  ' * depth}{s['name']} [{s['ms']} ms] {attrs}")
            self.tree(s["id"], depth + 1)

### The same governed run, instrumented: every model call and every gate decision becomes a span

`lst-traced-run`

In [ ]:
def run_traced_agent(user_input: str):
    with tracer.span("agent_run", user=user_input[:40]):
        messages = [
            {"role": "system", "content": (
                "You are a customer support agent. Use tools to check "
                "and issue refunds. If a tool reports that approval is "
                "required, tell the customer their request was "
                "escalated to a human reviewer.")},
            {"role": "user", "content": user_input},
        ]
        for step in range(POLICY["max_steps"]):
            with tracer.span("llm_call", model=CHAT_MODEL) as s:
                resp = client.chat.completions.create(
                    model=CHAT_MODEL, messages=messages,
                    tools=[status_tool, refund_tool],
                    tool_choice="auto", temperature=0.2, seed=42)
                s["attrs"]["input_tokens"] = resp.usage.prompt_tokens
                s["attrs"]["output_tokens"] = resp.usage.completion_tokens
            msg = resp.choices[0].message
            if not msg.tool_calls:
                return msg.content
            messages.append(
                {"role": "assistant", "tool_calls": msg.tool_calls})
            for tc in msg.tool_calls:
                args = json.loads(tc.function.arguments)
                decision, reason = gate(tc.function.name, args)
                with tracer.span("execute_tool",
                                 tool=tc.function.name,
                                 decision=decision):
                    if decision == "allow":
                        result = TOOL_IMPLEMENTATIONS[
                            tc.function.name](**args)
                    else:
                        result = {"status": f"blocked ({decision})",
                                  "reason": reason}
                messages.append(
                    {"role": "tool", "tool_call_id": tc.id,
                     "content": json.dumps(result)})
        return "Step budget exhausted; escalating to a human agent."

tracer = Tracer()
run_traced_agent("The item from order A-1002 arrived broken. "
                 "Please refund the 79.90 EUR now.")
tracer.tree()

### The Wilson score interval for a pass rate

`lst-wilson-interval`

In [ ]:
import math

def wilson(k, n, z=1.96):
    """Confidence interval for k successes in n cases."""
    if n == 0:
        return (0.0, 1.0)
    p = k / n
    denom = 1 + z**2 / n
    center = (p + z**2 / (2 * n)) / denom
    half = (z / denom) * math.sqrt(
        p * (1 - p) / n + z**2 / (4 * n**2)
    )
    return (max(0.0, center - half),
            min(1.0, center + half))

for n in (40, 100, 400, 1000):
    lo, hi = wilson(round(0.85 * n), n)
    print(f"n={n:>4}  85% pass  "
          f"[{lo:.1%}, {hi:.1%}]  "
          f"width {hi - lo:.1%}")